In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import copy
import random
from typing import Type, Any, Callable, Union, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import models, transforms
from torch.hub import load_state_dict_from_url

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
train_csv_fixed = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv"
val_csv_fixed   = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv"
test_csv_fixed  = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv"

print(os.path.exists(train_csv_fixed), train_csv_fixed)
print(os.path.exists(val_csv_fixed), val_csv_fixed)
print(os.path.exists(test_csv_fixed), test_csv_fixed)

True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv


In [15]:
train_csv_fixed = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv"
val_csv_fixed   = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv"
test_csv_fixed  = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv"

import os
print(os.path.exists(train_csv_fixed), train_csv_fixed)
print(os.path.exists(val_csv_fixed), val_csv_fixed)
print(os.path.exists(test_csv_fixed), test_csv_fixed)

True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv


In [16]:
train_df = pd.read_csv(train_csv_fixed)
val_df   = pd.read_csv(val_csv_fixed)
test_df  = pd.read_csv(test_csv_fixed)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print(train_df.head())
print(train_df.columns.tolist())
print("Train label min/max:", train_df.iloc[:, 2].min(), train_df.iloc[:, 2].max())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
['index', 'unit1_rgb', 'unit1_beam']
Train label min/max: 2 30


In [17]:
for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg False


In [18]:
class ImageBeamDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.iloc[1]
        label = int(row.iloc[2])

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, label

In [19]:
proc_pipe = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [20]:
import pandas as pd
import os

train_df = pd.read_csv(train_csv_fixed)
print(train_df.head())

for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_225

In [21]:
sample_img, sample_label = ImageBeamDataset(train_csv_fixed, transform=proc_pipe)[0]
print("Single sample image shape:", sample_img.shape)
print("Single sample label:", sample_label)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg'

In [22]:
import os

print("scenario23_dev exists:", os.path.exists("/content/dataset/scenario23_dev"))
print("camera_data exists:", os.path.exists("/content/dataset/scenario23_dev/unit1/camera_data"))

if os.path.exists("/content/dataset/scenario23_dev/unit1/camera_data"):
    files = os.listdir("/content/dataset/scenario23_dev/unit1/camera_data")[:10]
    print("Sample files:", files)

scenario23_dev exists: False
camera_data exists: False
